# Einlesen einer Aufnahme
Die Aufnahme ist eine CSV-Logdatei des Gesichtssystems. Alle Daten werden in
`aufnahme` als Liste von Dictionaries bereitgestellt. `merkmale` enthält die
numerischen Spalten als NumPy-Arrays; fehlende Werte sind `np.nan`.

Mit `LOGDATEI` kann eine Aufnahme ausgewählt werden. Bei `None` wird die zuletzt
geänderte Datei `logs/klassifikatoren_*.csv` im aktuellen Arbeitsverzeichnis geladen.

Benötigt werden NumPy und Matplotlib; die CSV-Datei wird mit der Python-
Standardbibliothek gelesen. Die Kamera und MediaPipe werden hier nicht benötigt.

Dieses Notebook verwendet **Case Based Reasoning (CBR)**: Eine Anfrage wird
mit gespeicherten, klassifizierten Fällen verglichen. Der Importcode ist mit
dem Evidenztheorie-Notebook identisch. Die Ähnlichkeitsberechnung benötigt
keine zusätzlichen Python-Pakete.


# Prüfungsleistung Künstliche Intelligeng - Teil II / Reichardt

Erstellen Sie einen Klassifikator für Emotionen aus kurzen Videoclips, die bereits durch ein gegebenes Vorverarbeitungssystem gelaufen sind und verschiedene Features bestimmt haben.

1. Schritt : Daten einlesen (eine Sequenz mit einer einzelnen Emotion)
2. Schritt : Festlegen einer Ähnlichkeitsfunktion
3. Schritt : Festlegen einer Struktur für eine Fallbeschreibung und umwandeln einer Messsequenz in einen Fall
4. Schritt : Aufbau einer Fallbasis (CBL)
5. Schritt : Klassifikation mit CBR

Zudem soll Ihr Klassifikator getestet werden, die Schritte begründet werden und die Ergebnisse interpretiert werden.

Nachfolgend ist ein Raster dafür mit Dummy Werten vorbereitet.

In [ ]:
from pathlib import Path
import csv
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Nur bei fehlenden Paketen einmalig aktivieren, danach ggf. Kernel neu starten:
# %pip install numpy matplotlib

LOGDATEI = None  # z. B. Path("logs/klassifikatoren_20261115_170000.csv")
LOGORDNER = Path("logs")

if LOGDATEI is None:
    dateien = list(LOGORDNER.glob("klassifikatoren_*.csv"))
    if not dateien:
        raise FileNotFoundError(
            f"Keine Logdatei in {LOGORDNER.resolve()}. LOGDATEI oder LOGORDNER anpassen."
        )
    logdatei = max(dateien, key=lambda p: (p.stat().st_mtime_ns, p.name))
else:
    logdatei = Path(LOGDATEI).expanduser()

with logdatei.open(encoding="utf-8-sig", newline="") as datei:
    reader = csv.DictReader(datei, delimiter=";")
    spalten = reader.fieldnames or []
    aufnahme = list(reader)

if not aufnahme:
    raise ValueError("Die ausgewählte Logdatei enthält keine Messzeilen.")
if "laufzeit_s" not in spalten:
    raise ValueError("Spalte laufzeit_s fehlt. Erwartet wird ein CSV-Log mit Semikolon-Trennung.")

merkmale = {}
for spalte in spalten:
    if spalte == "zeitstempel":
        continue
    werte = []
    numerisch = True
    for zeile in aufnahme:
        text = zeile.get(spalte)
        if text is None or not text.strip():
            werte.append(np.nan)
        else:
            try:
                werte.append(float(text))
            except ValueError:
                numerisch = False
                break
    if numerisch:
        merkmale[spalte] = np.asarray(werte, dtype=float)

if "laufzeit_s" not in merkmale or not np.isfinite(merkmale["laufzeit_s"]).any():
    raise ValueError("Die Aufnahme enthält keine gültige numerische Zeitachse.")

print("Aufnahme:", logdatei.resolve())
print("Messzeilen:", len(aufnahme))
print("Numerische Spalten:", ", ".join(merkmale))
print("Vorschau der ersten drei Messzeilen:")
display(aufnahme[:3])

# Beispiele für den späteren Zugriff:
# merkmale["auge_links_oeffnung"]
# merkmale["mund_oeffnung_geglaettet"]


# Ähnlichkeitsfunktion
Die Klassen sind `anger`, `joy`, `fear`, `surprise`, `disgust` und `sadness`.
Ein Fall hat die Form:

```python
{
    "id": "joy/aufnahme_01.csv",
    "problem": {"auge_links_oeffnung": 0.4, "mund_oeffnung_geglaettet": 0.3},
    "loesung": "joy",
    "quelle": "/pfad/zur/aufnahme_01.csv"
}
```

Die Problemrepräsentation ist **sparse**: Sie enthält nur tatsächlich vorhandene,
gültige Merkmale. Fehlende Attribute werden nicht mit 0 aufgefüllt. Eine echte
Messung 0 wird gespeichert. Fall-ID, Lösung und Quelldatei sind keine Vergleichsmerkmale.
Die Fallbasis ist eine Liste solcher Fälle, keine dichte Matrix. Sie können hier berechnete Features ergänzen.



In [ ]:
import json
from numbers import Real

KLASSEN = ("anger", "joy", "fear", "surprise", "disgust", "sadness")

# Ausgangskonfiguration: alle verwendeten Messmerkmale sind auf [0, 1] normiert.
# Gewichtung und lokale Ähnlichkeiten können als weitere Aufgabe angepasst werden.
GEWICHTE = {
    "auge_links_oeffnung": 1.0,
    "auge_rechts_oeffnung": 1.0,
    "stirn_horizontal": 1.0,
    "nasenwurzel_horizontal": 1.0,
    "nasenwurzel_vertikal": 1.0,
    "wange_links_vertikal": 1.0,
    "wange_rechts_vertikal": 1.0,
    "kopfneigung_wert": 1.0,
    "mund_oeffnung_geglaettet": 1.0,
}




# Klassifikation
Für die eingelesene Aufnahme wird ein Anfrageproblem erzeugt. Dann werden alle
Referenzfälle verglichen und nach globaler Ähnlichkeit sortiert. Die Lösung des
ähnlichsten Falls wird übernommen (1-Nearest-Neighbor, ohne zusätzliche Anpassung).
Bei einem Gleichstand zwischen verschiedenen Emotionen wird das Ergebnis als
mehrdeutig ausgewiesen. 

Die echte Fallbasis wird aus `fallbasis_emotionen.json` geladen. Diese müssen Sie zuvor aufbauen 
und gestalten.

Solange sie leer ist, können die ausdrücklich markierten **Dummy-Fälle** den Ablauf demonstrieren.
Sie sind keine trainierten oder validierten Emotionsprofile und werden später
nicht in die gelernte Fallbasis übernommen. 

Eine Aufnahme wird nicht mit ihrem eigenen gespeicherten Fall verglichen
(Prüfung über den aufgelösten Quelldateipfad). Kopien derselben Aufnahme unter
anderem Namen müssen bei der Versuchsplanung ebenfalls getrennt werden.
Die Diagrammbalken sind Ähnlichkeiten zu Fällen, keine Emotionswahrscheinlichkeiten.


In [ ]:


# Frei gewählte Demonstrationswerte, keine fachlich validierten Emotionsregeln.
# Absichtlich sparse und unterschiedlich vollständig.

DEMO_FALLBASIS = [
    {"id": "demo_anger", "problem": {"auge_links_oeffnung": 0.25,
        "auge_rechts_oeffnung": 0.25, "stirn_horizontal": 0.7,
        "nasenwurzel_vertikal": 0.8, "mund_oeffnung_geglaettet": 0.2},
        "loesung": "anger", "quelle": ""},
    {"id": "demo_joy", "problem": {"auge_links_oeffnung": 0.45,
        "auge_rechts_oeffnung": 0.45, "wange_links_vertikal": 0.6,
        "wange_rechts_vertikal": 0.6, "mund_oeffnung_geglaettet": 0.5},
        "loesung": "joy", "quelle": ""},
    {"id": "demo_fear", "problem": {"auge_links_oeffnung": 0.8,
        "auge_rechts_oeffnung": 0.8, "stirn_horizontal": 0.6,
        "nasenwurzel_horizontal": 0.4, "mund_oeffnung_geglaettet": 0.55},
        "loesung": "fear", "quelle": ""},
    {"id": "demo_surprise", "problem": {"auge_links_oeffnung": 0.95,
        "auge_rechts_oeffnung": 0.95, "stirn_horizontal": 0.75,
        "mund_oeffnung_geglaettet": 0.9},
        "loesung": "surprise", "quelle": ""},
    {"id": "demo_disgust", "problem": {"nasenwurzel_horizontal": 0.75,
        "nasenwurzel_vertikal": 0.75, "wange_links_vertikal": 0.7,
        "wange_rechts_vertikal": 0.7, "mund_oeffnung_geglaettet": 0.2},
        "loesung": "disgust", "quelle": ""},
    {"id": "demo_sadness", "problem": {"auge_links_oeffnung": 0.2,
        "auge_rechts_oeffnung": 0.2, "kopfneigung_wert": 0.4,
        "mund_oeffnung_geglaettet": 0.1},
        "loesung": "sadness", "quelle": ""},
]




# Case Based Learning
Alle CSV-Messungen unter `FALLVERZEICHNIS` werden rekursiv eingelesen.
**Eine ganze Messung wird zu einem Fall**, nicht jede einzelne CSV-Zeile.
Die Referenzemotion wird durch die Beschriftung der Aufnahme vorgegeben.

Vorgesehene Verzeichnisstruktur:

```text
faelle/
    anger/aufnahme_01.csv
    joy/aufnahme_02.csv
    fear/aufnahme_03.csv
    surprise/aufnahme_04.csv
    disgust/aufnahme_05.csv
    sadness/aufnahme_06.csv
```


In [ ]:
# Ihr code ...